This notebook is just a showcase of how we did the sampling. The CLI command `python -m mastodon_search choose-instances` does the same, but additionally probes the timelines API of all sampled instances, removes the ones that require a token or don't answer after a timeout. Because this takes a good amount of time, it is not part of this notebook.

In [ ]:
from pathlib import Path
from scipy.stats import lognorm
from numpy import exp

from mastodon_search.instance_data.analyze import Analyzer

In [ ]:
INSTANCE_DATA_PATH = Path('../data/instance_data.jsonl')
SAMPLE_OUTPUT_FILE_NAME = 'sampled_instances'
SAMPLE_OUTPUT_FILE_EXT = 'txt'
COLUMNS = ['total_users', 'monthly_users', 'total_statuses',
        'mean_weekly_statuses', 'mean_weekly_logins', 'mean_weekly_registrations']
NUM_SAMPLES = 1000

In [ ]:
cols_prob_measures = {
    col: lognorm
    for col in COLUMNS
}
with open(INSTANCE_DATA_PATH, 'r') as file:
    an = Analyzer(file)

In [ ]:
df = an.df
df

In [ ]:
df[cols_prob_measures.keys()].hist(log=True, bins=50)

In [ ]:
# Estimate probability distributions over activity columns
distributions = {
    col: dist.fit(df[col])
    for col, dist in cols_prob_measures.items()
}
distributions

In [ ]:
# Compute normalize activity score by dividing by the estimated probability.
for col, dist in cols_prob_measures.items():
    shape, location, scale = distributions[col]
    df[f"{col}_log_probability"] = dist.logpdf(df[col], shape, location, scale)

# Compute joint probability (under assumption of independence; using log probabilities for numerical stability)
df["log_probability"] = 0
for col in cols_prob_measures.keys():
    df["log_probability"] += df[f"{col}_log_probability"]

df.sort_values("log_probability", inplace=True)
df

In [ ]:
df["weight"] = exp(-df["log_probability"])
df_sample = df.sample(n=NUM_SAMPLES, replace=False, weights=df["weight"])
df_sample

In [ ]:
df_sample[cols_prob_measures.keys()].hist(log=True, bins=50)

In [ ]:
df_sample.sort_index(inplace=True)
# Full DataFrame. Maybe we want to have that data later.
df_sample.to_csv(Path(SAMPLE_OUTPUT_FILE_NAME + '_full_data.' + SAMPLE_OUTPUT_FILE_EXT))
sampled_instances = df_sample.reset_index()['instance']
# Raw instance list only.
sampled_instances.to_csv(Path(SAMPLE_OUTPUT_FILE_NAME + '.' + SAMPLE_OUTPUT_FILE_EXT), index=False, header=False)
sampled_instances